## Подключение

Запустите Theta Terminal v3 в macOS Terminal. Если он уже отвечает на порту 25503, используйте запущенный процесс.
```bash
"$(/usr/libexec/java_home -v 21)/bin/java" -jar ~/Downloads/ThetaTerminalv3.jar
```

Для Python нужны `pandas` и `requests`. При необходимости выполните `%pip install pandas requests`.

## Выгрузка всей цепочки

`download_options(тикер, дата_старта, наборы_данных, частота)` скачивает все страйки, call и put, включая истёкшие серии. Период — с указанной даты (для Standard не раньше `2016-01-01`) по вчерашний день по Нью-Йорку. CSV сохраняются в `theta_history/<тикер>/` по видам данных и датам. Если запрос разбит по экспирациям, дата экспирации добавляется к имени файла. При повторном запуске готовые CSV пропускаются.

Если ThetaData отвечает `Corrupt FIT data` на запрос часовой IV, функция запрашивает экспирации и контракты отдельно. Доступные строки остаются в дневном CSV, а контракты без данных перечислены в соседнем файле `.missing.txt`. При следующем запуске функция повторит запрос за этот день.

In [9]:
from contextlib import contextmanager
from io import StringIO
from pathlib import Path
from tempfile import NamedTemporaryFile

import pandas as pd
import requests

In [10]:
def download_options(symbol, start, datasets, interval):
    symbol = symbol.strip().upper()
    start = pd.Timestamp(start).normalize()
    end = pd.Timestamp.now(tz="America/New_York").date() - pd.Timedelta(days=1)
    if not pd.Timestamp("2016-01-01") <= start <= pd.Timestamp(end):
        raise ValueError("Укажите дату старта от 2016-01-01 до вчерашнего дня по Нью-Йорку.")

    daily = {"eod", "greeks/eod", "binomial_greeks/eod"}
    intraday = {"quote", "ohlc", "greeks/implied_volatility",
                "greeks/first_order", "binomial_greeks/first_order"}
    event_data = {"trade", "trade_quote"}
    subminute = {"tick", "10ms", "100ms", "500ms", "1s", "5s", "10s", "15s", "30s"}
    intervals = subminute | {"1m", "5m", "10m", "15m", "30m", "1h"}

    if isinstance(datasets, str):
        datasets = (datasets,)
    datasets = list(dict.fromkeys(datasets))
    unknown = set(datasets) - (daily | intraday | event_data | {"open_interest"})
    if unknown:
        raise ValueError(f"Неизвестные наборы данных: {', '.join(sorted(unknown))}")
    if intraday.intersection(datasets) and interval not in intervals:
        raise ValueError("Для котировок, OHLC и внутридневных греков выберите интервал из таблицы.")
    if "ohlc" in datasets and interval == "tick":
        raise ValueError("Для OHLC выберите интервал от 10ms до 1h. Для сделок используйте trade.")

    root = (Path("theta_history") / symbol).resolve()
    print(f"{symbol}: {start.date()} — {end} → {root}", flush=True)

    with requests.Session() as session:
        def get(endpoint, *, stream=False, **params):
            response = session.get(
                f"http://127.0.0.1:25503/v3/{endpoint}", params=params,
                stream=stream, timeout=(5, 180),
            )
            if not response.ok and response.status_code != 472:
                with response:
                    raise requests.HTTPError(
                        f"HTTP {response.status_code} {response.url}\n{response.text[:500]}",
                        response=response,
                    )
            return response

        def table(endpoint, **params):
            with get(endpoint, **params) as response:
                if response.status_code != 472:
                    return pd.read_csv(StringIO(response.text))
            return pd.DataFrame()

        @contextmanager
        def atomic_file(file):
            with NamedTemporaryFile(dir=file.parent, suffix=".part", delete=False) as output:
                partial = Path(output.name)
                try:
                    yield output
                    output.close()
                    partial.replace(file)
                finally:
                    partial.unlink(missing_ok=True)

        def save_csv(endpoint, file, **params):
            with get(endpoint, stream=True, **params) as response:
                if response.status_code == 472:
                    return False
                if "text/csv" not in response.headers.get("Content-Type", ""):
                    raise ValueError(f"Ожидался CSV: {response.url}")
                with atomic_file(file) as output:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        output.write(chunk)
            return True

        def corrupt_fit(error):
            response = error.response
            return (response is not None and response.status_code == 500
                    and "Corrupt FIT data" in response.text)

        def recover_hourly_iv(endpoint, file, params):
            contracts = table("option/list/contracts/quote", symbol=params["symbol"], date=params["date"])
            if contracts.empty:
                return False

            parts, gaps = [], []
            for expiration, chain in contracts.groupby("expiration"):
                query = dict(params, expiration=expiration)
                try:
                    frame = table(endpoint, **query)
                except requests.HTTPError as error:
                    if not corrupt_fit(error):
                        raise
                    frame = pd.DataFrame()

                keys = ["strike", "right"]
                available = set()
                if not frame.empty:
                    parts.append(frame)
                    available.update(frame[keys].itertuples(index=False, name=None))

                for strike, right in chain[keys].drop_duplicates().itertuples(index=False, name=None):
                    if (strike, right) in available:
                        continue
                    try:
                        contract = table(endpoint, **query, strike=strike, right=right.lower())
                    except requests.HTTPError as error:
                        if not corrupt_fit(error):
                            raise
                        gaps.append(f"{expiration} {strike:g} {right} — {error.response.text.strip()}")
                    else:
                        if contract.empty:
                            gaps.append(f"{expiration} {strike:g} {right} — ThetaData не вернула IV")
                        else:
                            parts.append(contract)

            report = file.with_suffix(".missing.txt")
            if gaps:
                with atomic_file(report) as output:
                    output.write(("Экспирация · страйк · тип · причина\n" + "\n".join(gaps) + "\n").encode())
                print(f"{file.stem}: без IV {len(gaps)} контракт(ов). Подробности в {report.name}", flush=True)
            if parts:
                with atomic_file(file) as output:
                    pd.concat(parts, ignore_index=True).to_csv(output, index=False)
            if not gaps:
                report.unlink(missing_ok=True)
            return bool(parts or gaps)

        closed = set()
        for year in range(start.year, end.year + 1):
            holidays = table("calendar/year_holidays", year=year)
            closed.update(holidays.loc[holidays["type"] == "full_close", "date"])

        for number, date in enumerate(pd.bdate_range(start, end), start=1):
            day = date.date().isoformat()
            if day in closed:
                continue

            for kind in datasets:
                params = {"symbol": symbol}
                if kind in daily:
                    params.update(start_date=day, end_date=day)
                else:
                    params["date"] = day
                if kind in intraday:
                    params["interval"] = interval
                if kind in event_data or (kind == "quote" and interval == "tick"):
                    params.update(start_time="00:00:00", end_time="23:59:59.999")

                name = kind.replace("/", "_")
                target = root / (f"{name}_{interval}" if kind in intraday else name)
                target.mkdir(parents=True, exist_ok=True)

                expirations = ["*"]
                if kind == "ohlc" or (kind in intraday and interval in subminute):
                    list_kind = "trade" if kind == "ohlc" else "quote"
                    contracts = table(f"option/list/contracts/{list_kind}", symbol=symbol, date=day)
                    expirations = sorted(contracts["expiration"].unique()) if not contracts.empty else []

                for expiration in expirations:
                    stem = day if expiration == "*" else f"{day}_{expiration}"
                    file = target / f"{stem}.csv"
                    report = file.with_suffix(".missing.txt")
                    if file.exists() and not report.exists():
                        continue

                    endpoint = f"option/history/{kind}"
                    try:
                        saved = save_csv(endpoint, file, expiration=expiration, **params)
                    except requests.HTTPError as error:
                        if (kind != "greeks/implied_volatility" or interval != "1h"
                                or not corrupt_fit(error)):
                            raise
                        print(f"{symbol} {day}: разбиваю запрос IV по экспирациям и контрактам", flush=True)
                        if not recover_hourly_iv(endpoint, file, params):
                            raise
                    else:
                        if saved:
                            report.unlink(missing_ok=True)

            if number % 100 == 0:
                print(f"{symbol}: обработано по {day}", flush=True)
    return root

### Запуск на всю глубину Standard

Укажите параметры и выставьте `RUN_FULL = True`. Выгрузка сразу покажет период и папку, затем будет сообщать о ходе работы.

| Что нужно | `DATASETS` | `INTERVAL` |
| --- | --- | --- |
| Дневные цены, объём и OI | `("eod", "open_interest")` | `"1d"` |
| Каждая сделка или сделка с NBBO | `"trade"` или `"trade_quote"` | `"tick"` |
| Каждое обновление NBBO | `"quote"` | `"tick"` |
| Внутридневные котировки и бары | `("quote", "ohlc")` | например, `"1m"` |
| Дневные IV и греки | `("greeks/eod", "binomial_greeks/eod")` | `"1d"` |
| Внутридневная IV по bid, mid и ask | `"greeks/implied_volatility"` | например, `"1m"` |
| Внутридневные греки первого порядка | `("greeks/first_order", "binomial_greeks/first_order")` | например, `"5m"` |

Доступные интервалы — `10ms`, `100ms`, `500ms`, `1s`, `5s`, `10s`, `15s`, `30s`, `1m`, `5m`, `10m`, `15m`, `30m`, `1h`. Для котировок, внутридневной IV и греков можно также выбрать `"tick"`. Бары OHLC требуют фиксированного интервала. Для EOD, OI и сделок частоту определяет сам набор данных.

Чтобы выгрузить дневные отчёты, OI, сделки и тики NBBO вместе, укажите `DATASETS = ("eod", "open_interest", "trade", "quote")` и `INTERVAL = "tick"`. Сделки и тики NBBO запрашиваются за весь день по Нью-Йорку. Остальные внутридневные данные используют окно API 09:30–16:00 ET.

[Options Standard](https://thetadata.net/docs/Articles/Getting-Started/Subscriptions.html#options-data) · [Интервалы](https://thetadata.net/docs/operations/option_history_quote.html#query-parameters)

In [11]:
RUN_FULL = True
SYMBOL = "BA"
START = "2020-08-07"
DATASETS = ("greeks/implied_volatility")
INTERVAL = "1h"

if RUN_FULL:
    archive = download_options(SYMBOL, START, DATASETS, INTERVAL)
    print("Архив:", archive)

BA: 2020-08-07 — 2026-11-16 → /Users/stepansapunkov/Documents/ВКР/Чистовик_скриптов/Data/Options/theta_history/BA
BA: обработано по 2020-12-24
BA 2021-02-17: разбиваю запрос IV по экспирациям и контрактам
2021-02-17: без IV 1 контракт(ов). Подробности в 2021-02-17.missing.txt
BA: обработано по 2021-05-13
BA 2021-06-24: разбиваю запрос IV по экспирациям и контрактам
2021-06-24: без IV 1 контракт(ов). Подробности в 2021-06-24.missing.txt
BA 2021-09-07: разбиваю запрос IV по экспирациям и контрактам
2021-09-07: без IV 1 контракт(ов). Подробности в 2021-09-07.missing.txt
BA: обработано по 2021-09-30
BA: обработано по 2022-02-17
BA: обработано по 2022-07-07
BA 2023-01-03: разбиваю запрос IV по экспирациям и контрактам
2023-01-03: без IV 1 контракт(ов). Подробности в 2023-01-03.missing.txt
BA: обработано по 2023-04-13
BA: обработано по 2023-08-31
BA: обработано по 2024-01-18
BA: обработано по 2024-06-06
BA: обработано по 2024-10-24
BA: обработано по 2025-03-13
BA: обработано по 2025-07-31
BA